# Module 6 Homework

In this homework we'll put what we learned about Spark in practice.

For this homework we will be using the Yellow 2025-11 data from the official website:

```bash
wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
```

In [1]:
!curl -L -O https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 14 67.8M   14 10.1M    0     0  9580k      0  0:00:07  0:00:01  0:00:06 9584k
 35 67.8M   35 24.1M    0     0  11.8M      0  0:00:05  0:00:02  0:00:03 11.8M
 59 67.8M   59 40.3M    0     0  13.3M      0  0:00:05  0:00:03  0:00:02 13.3M
 84 67.8M   84 57.2M    0     0  14.2M      0  0:00:04  0:00:04 --:--:-- 14.2M
100 67.8M  100 67.8M    0     0  14.5M      0  0:00:04  0:00:04 --:--:-- 14.7M


In [2]:
!sh -c "mkdir -p data/raw/yellow/2025/11/ && mv yellow_tripdata_2025-11.parquet data/raw/yellow/2025/11/"

## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

> [!NOTE]
> To install PySpark follow this [guide](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/06-batch/setup/)

In [3]:
from pyspark.sql import SparkSession

# 1. Crear la sesión
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [4]:
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- 25MB
- 75MB
- 100MB

In [5]:
year = 2025
month = 11

input_path = f'data/raw/yellow/{year}/{month:02d}/'
output_path = f'data/pq/yellow/{year}/{month:02d}/'

df = spark.read.parquet(input_path)

df = df.repartition(4)

df.write.parquet(output_path, mode="overwrite")

In [6]:
!sh -c "ls -lh data/pq/yellow/2025/11/*.parquet"

-rw-r--r-- 1 ANDRE 197121 25M Mar 10 00:01 data/pq/yellow/2025/11/part-00000-5ed25fd8-08ca-4024-90c1-60af04896021-c000.snappy.parquet
-rw-r--r-- 1 ANDRE 197121 25M Mar 10 00:01 data/pq/yellow/2025/11/part-00001-5ed25fd8-08ca-4024-90c1-60af04896021-c000.snappy.parquet
-rw-r--r-- 1 ANDRE 197121 25M Mar 10 00:01 data/pq/yellow/2025/11/part-00002-5ed25fd8-08ca-4024-90c1-60af04896021-c000.snappy.parquet
-rw-r--r-- 1 ANDRE 197121 25M Mar 10 00:01 data/pq/yellow/2025/11/part-00003-5ed25fd8-08ca-4024-90c1-60af04896021-c000.snappy.parquet


## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610
- 102,340
- 162,604
- 225,768

In [7]:
from pyspark.sql.functions import to_date

df_repartitioned = spark.read \
        .option("header", "true") \
        .parquet("data/pq/yellow/2025/11/")
df_repartitioned.filter(to_date(df_repartitioned.tpep_pickup_datetime) == "2025-11-15").count()

162604

## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

- 22.7
- 58.2
- 90.6
- 134.5


In [8]:
from pyspark.sql.functions import unix_timestamp, round, max

df_repartitioned\
    .withColumn("trip_duration", \
                round((unix_timestamp("tpep_dropoff_datetime")-unix_timestamp("tpep_pickup_datetime")) /3600 , 2)) \
    .select(max("trip_duration")).show()

+------------------+
|max(trip_duration)|
+------------------+
|             90.65|
+------------------+



## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- 4040
- 8080

In [9]:
# 4040

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

- Governor's Island/Ellis Island/Liberty Island
- Arden Heights
- Rikers Island
- Jamaica Bay

If multiple answers are correct, select any


In [10]:
!sh -c "mkdir -p data/raw/misc && curl -o data/raw/misc/taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 12331  100 12331    0     0  26960      0 --:--:-- --:--:-- --:--:-- 27041


In [11]:
df_yellow = spark.read \
        .option("header", "true") \
        .parquet("data/pq/yellow/2025/11/")
df_yellow.limit(5).toPandas()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2025-11-02 08:11:08,2025-11-02 08:15:21,1,1.24,1,N,186,230,1,7.9,0.0,0.5,2.53,0.0,1.0,15.18,2.5,0.0,0.75
1,2,2025-11-06 14:01:48,2025-11-06 14:25:53,2,1.84,1,N,164,237,2,20.5,0.0,0.5,0.00,0.0,1.0,25.25,2.5,0.0,0.75
2,2,2025-11-07 16:53:08,2025-11-07 17:10:10,1,1.15,1,N,186,161,1,14.9,2.5,0.5,0.04,0.0,1.0,22.19,2.5,0.0,0.75
3,2,2025-11-09 10:55:05,2025-11-09 10:58:57,1,0.55,1,N,68,246,1,5.8,0.0,0.5,2.11,0.0,1.0,12.66,2.5,0.0,0.75
4,2,2025-11-03 13:35:09,2025-11-03 13:47:48,1,0.91,1,N,90,164,1,12.1,0.0,0.5,3.37,0.0,1.0,20.22,2.5,0.0,0.75


In [12]:
df_zones = spark.read \
        .option("header", "true") \
        .csv("data/raw/misc/")
df_zones.limit(5).toPandas()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [13]:
from pyspark.sql.functions import col

df_join = df_repartitioned.alias('trips') \
    .join(df_zones.alias('zones'), 
          col('trips.PULocationID') == col('zones.LocationID'), 'left')\
    .select("trips.*", "zones.Zone")
# Ahora puedes seleccionar usando el alias
df_join.groupBy("Zone").count().orderBy(col("count").asc()).limit(5).toPandas()

,Zone,count
0,Governor's Island/Ellis Island/Liberty Island,1
1,Eltingville/Annadale/Prince's Bay,1
2,Arden Heights,1
3,Port Richmond,3
4,Rikers Island,4
